In [1]:
import fitz  # PyMuPDF

PDF_PATH = r"/home/son/Desktop/rag/app/data/rag_docs.pdf"


def read_pdf(path: str):
    doc = fitz.open(path)

    print(f"تعداد صفحات: {doc.page_count}")
    print("=" * 80)

    for page_number, page in enumerate(doc, start=1):
        text = page.get_text()

        print(f"\n===== Page {page_number} =====\n")
        print(text[:1000])      # فقط 1000 کاراکتر اول
        print("\n" + "-" * 80)


if __name__ == "__main__":
    read_pdf(PDF_PATH)

تعداد صفحات: 13

===== Page 1 =====

۱ .
 مقدمه و پیشینه 
در دنیای امروز، مدل های زبانی بزرگ (LLM) به یکی از پایه
های اصلی نرم افزارهای هوشمند تبدیل
شده
اند. این مدل
ها قادرند متون طبیعی را درک کرده، آن
ها را تحلیل کنند و پاسخ هایی دقیق و روان
 تولید نمایند. با این حال، یکی از بزرگ
ترین چالش
های این مدل
ها آن است که دانش آن ها به زمان
آموزش محدود می
شود و دسترسی مستقیمی به داده
های جدید، اختصاصی یا سازمانی ندارند.
 
 
برای رفع این محدودیت، رویکرد «بازیابی-
 افزوده تولید» یاRAG (Retrieval-Augmented 
Generation) 
 مطرح شد. این رویکرد ترکیبی از دو مرحله اصلی است:
 
1. در مرحله اول، اطالعات مرتبط از یک پایگاه دانش بیرونی بازیابی می
شود.
 
2. در مرحله دوم، مدل زبانی از این اطالعات بازیابی
شده به عنوان زمینه (Context) استفاده می کند
تا پاسخ نهایی را تولید نماید.
 
 
معماری RAG 
 نخستین بار توسط
لوئیس و همکارانش 
 در سال۲۰۲۰
 
 معرفی شد و از آن زمان تاکنون
تحوالت گسترده ای را تجربه کرده است. امروزه این معماری در طیف وسیعی از کاربردها، از جمله
پرسش و پاسخ سازمانی، چت
بات
های پشتیبانی مشتری، تح

In [2]:
import fitz
import re
from dataclasses import dataclass, field

@dataclass
class Chunk:
    chunk_id: str
    type: str          # "parent" یا "child"
    title: str
    content: str
    parent_id: str | None = None
    page: int = 0

def extract_blocks_with_style(path: str):
    """استخراج بلاک‌ها با اطلاعات فونت"""
    doc = fitz.open(path)
    blocks = []

    for page_num, page in enumerate(doc, start=1):
        # dict mode اطلاعات کامل‌تری میده
        page_dict = page.get_text("dict")

        for block in page_dict["blocks"]:
            if block["type"] != 0:  # فقط text block
                continue

            block_text = ""
            max_font_size = 0

            for line in block["lines"]:
                for span in line["spans"]:
                    block_text += span["text"] + " "
                    max_font_size = max(max_font_size, span["size"])

            block_text = block_text.strip()
            if not block_text:
                continue

            blocks.append({
                "text": block_text,
                "font_size": round(max_font_size, 1),
                "page": page_num
            })

    return blocks

def detect_structure(blocks: list) -> tuple[float, float]:
    """پیدا کردن threshold فونت برای parent و child"""
    sizes = [b["font_size"] for b in blocks]
    sizes_sorted = sorted(set(sizes), reverse=True)

    print("فونت‌سایزهای موجود:", sizes_sorted[:10])

    # معمولاً:
    # بزرگ‌ترین سایز = Parent (عنوان اصلی)
    # سایز دوم = Child (زیرعنوان)
    # بقیه = body text
    parent_threshold = sizes_sorted[0] if len(sizes_sorted) > 0 else 16
    child_threshold = sizes_sorted[1] if len(sizes_sorted) > 1 else 13

    return parent_threshold, child_threshold


In [3]:
blocks = extract_blocks_with_style(r"D:\rag\app\data\rag_docs.pdf")

# نمایش فونت‌سایز هر بلاک
for b in blocks[:30]:
    print(f"[{b['font_size']}pt | p{b['page']}] {b['text'][:60]}")

[20.0pt | p1] ۱  .  مقدمه و پیشینه
[16.0pt | p1] د ر دنیای امروز، مدل های زبانی بزرگ (LLM) به یکی از پایه های
[16.0pt | p1] شده اند. این مدل ها قادرند متون طبیعی را درک کرده، آن ها را 
[16.0pt | p1] تولید نمایند. با این حال، یکی از بزرگ ترین چالش های این مدل 
[16.0pt | p1] آموزش محدود می شود و دسترسی مستقیمی به داده های جدید، اختصاص
[16.0pt | p1] برای رفع این محدودیت، رویکرد «بازیابی-  افزوده تولید» یاRAG 
[16.0pt | p1] Generation)   مطرح شد. این رویکرد ترکیبی از دو مرحله اصلی اس
[16.0pt | p1] 1.   در مرحله اول، اطالعات مرتبط از یک پایگاه دانش بیرونی با
[16.0pt | p1] 2.   در مرحله دوم، مدل زبانی از این اطالعات بازیابی شده به ع
[16.0pt | p1] تا پاسخ نهایی را تولید نماید.
[16.0pt | p1] معماری RAG   نخستین بار توسط لوئیس و همکارانش   در سال۲۰۲۰  
[16.0pt | p1] تحوالت گسترده ای را تجربه کرده است. امروزه این معماری در طیف
[16.0pt | p1] پرسش و پاسخ سازمانی، چت بات های پشتیبانی مشتری، تحلیل اسناد 
[16.0pt | p1] هوش تجاری (BI) مورد استفاده قرار می گیرد.
[20.0pt | p2] ۲ . اجزای اصلی یک سیستم   

In [5]:
import fitz
import re
from dataclasses import dataclass, field

@dataclass
class Chunk:
    chunk_id: str
    type: str           # "parent" یا "child"
    title: str
    content: str
    parent_id: str | None = None
    page: int = 0

# --- Pattern های شناسایی عنوان ---

# Parent: ۱ . عنوان  یا  ۲ . عنوان  (عدد فارسی + نقطه)
PARENT_PATTERN = re.compile(
    r'^[\u06F0-\u06F9۰-۹\d]+\s*[\.\．]\s+\S'
)

# Child numeric: ۲.۱ عنوان  یا  2.1 عنوان
CHILD_NUMERIC_PATTERN = re.compile(
    r'^[\u06F0-\u06F9۰-۹\d]+[\.\．][\u06F0-\u06F9۰-۹\d]+\s+\S'
)

# Child alphabetic: الف)  ب)  ج)  یا  a)  b)
CHILD_ALPHA_PATTERN = re.compile(
    r'^(الف|ب|پ|ت|ث|ج|چ|[a-zA-Z])\s*[\)\]]\s+\S'
)

def classify_block(text: str, font_size: float) -> str:
    """تشخیص نوع بلاک"""
    text = text.strip()

    if font_size >= 19.0:
        return "parent"

    if CHILD_NUMERIC_PATTERN.match(text):
        return "child"

    if CHILD_ALPHA_PATTERN.match(text):
        return "child"

    return "body"


def extract_blocks_with_style(path: str) -> list[dict]:
    doc = fitz.open(path)
    blocks = []

    for page_num, page in enumerate(doc, start=1):
        page_dict = page.get_text("dict")

        for block in page_dict["blocks"]:
            if block["type"] != 0:
                continue

            block_text = ""
            max_font_size = 0

            for line in block["lines"]:
                for span in line["spans"]:
                    block_text += span["text"] + " "
                    max_font_size = max(max_font_size, span["size"])

            block_text = block_text.strip()
            if not block_text:
                continue

            blocks.append({
                "text": block_text,
                "font_size": round(max_font_size, 1),
                "page": page_num,
                "type": classify_block(block_text, max_font_size)
            })

    return blocks


def build_chunks(path: str) -> list[Chunk]:
    blocks = extract_blocks_with_style(path)

    chunks: list[Chunk] = []
    current_parent: Chunk | None = None
    current_child: Chunk | None = None
    parent_counter = 0
    child_counter = 0

    def flush_child():
        nonlocal current_child
        if current_child:
            chunks.append(current_child)
            current_child = None

    for block in blocks:
        btype = block["type"]
        text  = block["text"]
        page  = block["page"]

        if btype == "parent":
            flush_child()
            if current_parent:
                chunks.append(current_parent)

            parent_counter += 1
            current_parent = Chunk(
                chunk_id=f"parent_{parent_counter}",
                type="parent",
                title=text,
                content=text,
                page=page
            )

        elif btype == "child":
            flush_child()
            child_counter += 1
            current_child = Chunk(
                chunk_id=f"child_{child_counter}",
                type="child",
                title=text,
                content=text,
                parent_id=current_parent.chunk_id if current_parent else None,
                page=page
            )

        else:  # body
            if current_child:
                current_child.content += "\n" + text
            elif current_parent:
                current_parent.content += "\n" + text

    flush_child()
    if current_parent:
        chunks.append(current_parent)

    return chunks

In [6]:
chunks = build_chunks(r"D:\rag\app\data\rag_docs.pdf")

for c in chunks:
    indent = "  └─" if c.type == "child" else "■"
    print(f"{indent} [{c.type}] {c.title[:60]}")
    print(f"     parent_id={c.parent_id} | page={c.page} | len={len(c.content)}")
    print()

■ [parent] ۱  .  مقدمه و پیشینه
     parent_id=None | page=1 | len=1030

  └─ [child] ۲.۱  پردازش و بارگذاری اسناد   (Document Ingestion)
     parent_id=parent_2 | page=2 | len=417

  └─ [child] ۲.۲  تقسیم بندی متن (Chunking)
     parent_id=parent_2 | page=2 | len=227

  └─ [child] الف) تقسیم بندی با اندازه ثابت (Fixed-size Chunking)
     parent_id=parent_2 | page=2 | len=245

  └─ [child] ب) تقسیم بندی معنایی (Semantic Chunking)
     parent_id=parent_2 | page=3 | len=242

  └─ [child] ج) تقسیم بندی با هم پوشانی (Overlapping Chunks)
     parent_id=parent_2 | page=3 | len=276

  └─ [child] ۲.۳    تبدیل به بردار (Embedding)
     parent_id=parent_2 | page=3 | len=637

  └─ [child] ۲.۴  پایگاه داده برداری (Vector Store)
     parent_id=parent_2 | page=4 | len=1112

  └─ [child] ۲.۶    تولید پاسخ (Generation)
     parent_id=parent_2 | page=5 | len=363

■ [parent] ۲ . اجزای اصلی یک سیستم   RAG
     parent_id=None | page=2 | len=134

  └─ [child] ۳.۱    چندریختی متن فارسی
     parent_id=parent

In [3]:
import re

CHILD_NUMERIC_PATTERN = re.compile(
    r'^[\u06F0-\u06F9۰-۹\d]+[\.\．][\u06F0-\u06F9۰-۹\d]+\s+\S'
)

def classify_block(text: str, font_size: float) -> str:
    text = text.strip()
    if font_size >= 19.0:
        return "parent"
    if CHILD_NUMERIC_PATTERN.match(text):
        return "child"
    return "body"  # الف) ب) و بقیه همه body

In [7]:
def build_chunks(path: str) -> list[Chunk]:
    blocks = extract_blocks_with_style(path)

    chunks: list[Chunk] = []
    current_parent: Chunk | None = None
    current_child: Chunk | None = None
    parent_counter = 0
    child_counter = 0

    def flush_child():
        nonlocal current_child
        if current_child:
            chunks.append(current_child)
            current_child = None

    def get_or_create_auto_child(page: int) -> Chunk:
        """اگه parent بدون child داره body می‌گیره، یه child خودکار بساز"""
        nonlocal current_child, child_counter
        if current_child is None and current_parent is not None:
            child_counter += 1
            current_child = Chunk(
                chunk_id=f"child_{child_counter}",
                type="child",
                title=current_parent.title,  # عنوان همون parent
                content="",
                parent_id=current_parent.chunk_id,
                page=page
            )
        return current_child

    for block in blocks:
        btype = block["type"]
        text  = block["text"]
        page  = block["page"]

        if btype == "parent":
            flush_child()
            if current_parent:
                chunks.append(current_parent)

            parent_counter += 1
            current_parent = Chunk(
                chunk_id=f"parent_{parent_counter}",
                type="parent",
                title=text,
                content=text,
                page=page
            )

        elif btype == "child":
            flush_child()
            child_counter += 1
            current_child = Chunk(
                chunk_id=f"child_{child_counter}",
                type="child",
                title=text,
                content=text,
                parent_id=current_parent.chunk_id if current_parent else None,
                page=page
            )

        else:  # body
            target = get_or_create_auto_child(page)
            if target:
                target.content += "\n" + text

    flush_child()
    if current_parent:
        chunks.append(current_parent)

    return chunks

In [8]:
chunks = build_chunks(r"D:\rag\app\data\rag_docs.pdf")

for c in chunks:
    indent = "  └─" if c.type == "child" else "■"
    print(f"{indent} [{c.type}] {c.title[:60]}")
    print(f"     parent_id={c.parent_id} | page={c.page} | len={len(c.content)}")
    print()

  └─ [child] ۱  .  مقدمه و پیشینه
     parent_id=parent_1 | page=1 | len=1010

■ [parent] ۱  .  مقدمه و پیشینه
     parent_id=None | page=1 | len=20

  └─ [child] ۲ . اجزای اصلی یک سیستم   RAG
     parent_id=parent_2 | page=2 | len=105

  └─ [child] ۲.۱  پردازش و بارگذاری اسناد   (Document Ingestion)
     parent_id=parent_2 | page=2 | len=417

  └─ [child] ۲.۲  تقسیم بندی متن (Chunking)
     parent_id=parent_2 | page=2 | len=227

  └─ [child] الف) تقسیم بندی با اندازه ثابت (Fixed-size Chunking)
     parent_id=parent_2 | page=2 | len=245

  └─ [child] ب) تقسیم بندی معنایی (Semantic Chunking)
     parent_id=parent_2 | page=3 | len=242

  └─ [child] ج) تقسیم بندی با هم پوشانی (Overlapping Chunks)
     parent_id=parent_2 | page=3 | len=276

  └─ [child] ۲.۳    تبدیل به بردار (Embedding)
     parent_id=parent_2 | page=3 | len=637

  └─ [child] ۲.۴  پایگاه داده برداری (Vector Store)
     parent_id=parent_2 | page=4 | len=1112

  └─ [child] ۲.۶    تولید پاسخ (Generation)
     parent_id=parent

In [9]:
from dataclasses import dataclass, field
import fitz
import re

@dataclass
class ChildChunk:
    id: str
    title: str
    content: str
    parent_id: str

@dataclass
class ParentChunk:
    id: str
    title: str
    content: str
    children: list[ChildChunk] = field(default_factory=list)


CHILD_NUMERIC_PATTERN = re.compile(
    r'^[\u06F0-\u06F9۰-۹\d]+[\.\．][\u06F0-\u06F9۰-۹\d]+\s+\S'
)

def classify_block(text: str, font_size: float) -> str:
    if font_size >= 19.0:
        return "parent"
    if CHILD_NUMERIC_PATTERN.match(text.strip()):
        return "child"
    return "body"

def extract_blocks(path: str) -> list[dict]:
    doc = fitz.open(path)
    blocks = []
    for page_num, page in enumerate(doc, start=1):
        for block in page.get_text("dict")["blocks"]:
            if block["type"] != 0:
                continue
            text, max_size = "", 0
            for line in block["lines"]:
                for span in line["spans"]:
                    text += span["text"] + " "
                    max_size = max(max_size, span["size"])
            text = text.strip()
            if text:
                blocks.append({"text": text, "font_size": round(max_size, 1),
                                "page": page_num, "type": classify_block(text, max_size)})
    return blocks

def build_chunks(path: str) -> list[ParentChunk]:
    blocks = extract_blocks(path)

    parents: list[ParentChunk] = []
    current_parent: ParentChunk | None = None
    current_child: ChildChunk | None = None
    parent_counter = 0
    child_counter = 0

    def flush_child():
        nonlocal current_child
        if current_child and current_parent:
            current_parent.children.append(current_child)
            current_parent.content += "\n" + current_child.content
            current_child = None

    def get_or_create_auto_child(text: str):
        nonlocal current_child, child_counter
        if current_child is None and current_parent is not None:
            child_counter += 1
            current_child = ChildChunk(
                id=f"{current_parent.id}.0",
                title=current_parent.title,
                content="",
                parent_id=current_parent.id
            )

    for block in blocks:
        btype, text = block["type"], block["text"]

        if btype == "parent":
            flush_child()
            if current_parent:
                parents.append(current_parent)
            parent_counter += 1
            current_parent = ParentChunk(
                id=str(parent_counter),
                title=text,
                content=""
            )

        elif btype == "child":
            flush_child()
            child_counter += 1
            current_child = ChildChunk(
                id=f"{current_parent.id}.{child_counter}" if current_parent else str(child_counter),
                title=text,
                content=text,
                parent_id=current_parent.id if current_parent else ""
            )

        else:  # body
            get_or_create_auto_child(text)
            if current_child:
                current_child.content += "\n" + text

    flush_child()
    if current_parent:
        parents.append(current_parent)

    return parents

In [2]:
def build_chunks(path: str) -> list[ParentChunk]:
    blocks = extract_blocks(path)

    parents: list[ParentChunk] = []
    current_parent: ParentChunk | None = None
    current_child: ChildChunk | None = None
    parent_counter = 0
    child_counter = 0  # ریست میشه برای هر parent

    def flush_child():
        nonlocal current_child
        if current_child and current_parent:
            current_parent.children.append(current_child)
            current_parent.content += "\n" + current_child.content
            current_child = None

    def get_or_create_auto_child(text: str):
        nonlocal current_child, child_counter
        if current_child is None and current_parent is not None:
            child_counter += 1
            current_child = ChildChunk(
                id=f"{current_parent.id}.{child_counter}",
                title=current_parent.title,
                content="",
                parent_id=current_parent.id
            )

    for block in blocks:
        btype, text = block["type"], block["text"]

        if btype == "parent":
            flush_child()
            if current_parent:
                parents.append(current_parent)
            parent_counter += 1
            child_counter = 0  # ← اینجا ریست میشه
            current_parent = ParentChunk(
                id=str(parent_counter),
                title=text,
                content=""
            )

        elif btype == "child":
            flush_child()
            child_counter += 1
            current_child = ChildChunk(
                id=f"{current_parent.id}.{child_counter}" if current_parent else str(child_counter),
                title=text,
                content=text,
                parent_id=current_parent.id if current_parent else ""
            )

        else:  # body
            get_or_create_auto_child(text)
            if current_child:
                current_child.content += "\n" + text

    flush_child()
    if current_parent:
        parents.append(current_parent)

    return parents

In [11]:
def build_chunks(path: str) -> list[ParentChunk]:
    blocks = extract_blocks(path)

    parents: list[ParentChunk] = []

    current_parent: ParentChunk | None = None
    current_child: ChildChunk | None = None

    parent_counter = 0
    child_counter = 0  # برای هر Parent از 1 شروع می‌شود

    def flush_child():
        nonlocal current_child

        if current_parent is None or current_child is None:
            return

        current_parent.children.append(current_child)
        current_parent.content += "\n" + current_child.content
        current_child = None

    def get_or_create_auto_child():
        nonlocal current_child, child_counter

        if current_parent is None or current_child is not None:
            return

        child_counter += 1

        current_child = ChildChunk(
            id=f"{current_parent.id}.{child_counter}",
            title=current_parent.title,
            content="",
            parent_id=current_parent.id,
        )

    for block in blocks:
        btype = block["type"]
        text = block["text"]

        # ---------------- Parent ----------------
        if btype == "parent":
            flush_child()

            if current_parent is not None:
                parents.append(current_parent)

            parent_counter += 1
            child_counter = 0  # ریست شماره Childها برای Parent جدید

            current_parent = ParentChunk(
                id=str(parent_counter),
                title=text,
                content="",
            )

        # ---------------- Child ----------------
        elif btype == "child":
            if current_parent is None:
                continue

            flush_child()

            child_counter += 1

            current_child = ChildChunk(
                id=f"{current_parent.id}.{child_counter}",
                title=text,
                content=text,
                parent_id=current_parent.id,
            )

        # ---------------- Body ----------------
        else:
            if current_parent is None:
                continue

            get_or_create_auto_child()

            current_child.content += "\n" + text

    flush_child()

    if current_parent is not None:
        parents.append(current_parent)

    return parents

In [9]:
import re
from app.models.Chunks import ChildChunk, ParentChunk
import fitz

CHILD_NUMERIC_PATTERN = re.compile(
    r'^[\u06F0-\u06F9۰-۹\d]+[\.\．][\u06F0-\u06F9۰-۹\d]+\s+\S'
)

def classify_block(text: str, font_size: float) -> str:
    text = text.strip()
    if font_size >= 19.0:
        return "parent"
    if CHILD_NUMERIC_PATTERN.match(text):
        return "child"
    return "body"  # الف) ب) و بقیه همه body

TITLE_PREFIX_PATTERN = re.compile(
    r"^\s*[\d۰-۹]+(?:[\.．][\d۰-۹]+)*\s*[\.．]?\s*"
)

def clean_title(title: str) -> str:
    """شماره ابتدای عنوان (مثل ۵. یا ۳.۲.) را حذف می‌کند."""
    return TITLE_PREFIX_PATTERN.sub("", title).strip()



def extract_blocks(path: str) -> list[dict]:
    doc = fitz.open(path)
    blocks = []
    for page_num, page in enumerate(doc, start=1):
        for block in page.get_text("dict")["blocks"]:
            if block["type"] != 0:
                continue
            text, max_size = "", 0
            for line in block["lines"]:
                for span in line["spans"]:
                    text += span["text"] + " "
                    max_size = max(max_size, span["size"])
            text = text.strip()
            if text:
                blocks.append({"text": text, "font_size": round(max_size, 1),
                                "page": page_num, "type": classify_block(text, max_size)})
    return blocks



def build_chunks(path: str) -> list[ParentChunk]:
    blocks = extract_blocks(path)

    parents: list[ParentChunk] = []

    current_parent: ParentChunk | None = None
    current_child: ChildChunk | None = None

    parent_counter = 0
    child_counter = 0

    # متن‌های قبل از اولین Child واقعی
    pending_body: list[str] = []

    # آیا این Parent حداقل یک Child واقعی دارد؟
    has_real_child = False

    def flush_child():
        nonlocal current_child

        if current_parent is None or current_child is None:
            return

        current_parent.children.append(current_child)
        current_parent.content += "\n" + current_child.content
        current_child = None

    def flush_pending_as_intro():
        """متن‌های قبل از اولین Child واقعی را به عنوان مقدمه ذخیره می‌کند."""
        nonlocal child_counter

        if current_parent is None or not pending_body:
            return

        child_counter += 1

        intro = ChildChunk(
            id=f"{current_parent.id}.{child_counter}",
            title=f"مقدمه: {current_parent.title}",
            content="\n".join(pending_body),
            parent_id=current_parent.id,
        )

        current_parent.children.append(intro)
        current_parent.content += "\n" + intro.content

        pending_body.clear()

    def flush_pending_as_single_child():
        """اگر Parent هیچ Child واقعی نداشت، کل متن را یک Child می‌کند."""
        nonlocal child_counter

        if current_parent is None or not pending_body:
            return

        child_counter += 1

        only_child = ChildChunk(
            id=f"{current_parent.id}.{child_counter}",
            title=current_parent.title,
            content="\n".join(pending_body),
            parent_id=current_parent.id,
        )

        current_parent.children.append(only_child)
        current_parent.content += "\n" + only_child.content

        pending_body.clear()

    for block in blocks:

        btype = block["type"]
        text = block["text"]

        # ---------------- Parent ----------------
        if btype == "parent":

            flush_child()

            if current_parent is not None:

                if not has_real_child:
                    flush_pending_as_single_child()

                parents.append(current_parent)

            parent_counter += 1
            child_counter = 0

            current_parent = ParentChunk(
                id=str(parent_counter),
                title=clean_title(text),   # ← عنوان Parent بدون شماره
                content="",
            )

            pending_body.clear()
            has_real_child = False

        # ---------------- Child ----------------
        elif btype == "child":

            if current_parent is None:
                continue

            if not has_real_child:
                flush_pending_as_intro()

            has_real_child = True

            flush_child()

            child_counter += 1

            current_child = ChildChunk(
                id=f"{current_parent.id}.{child_counter}",
                title=clean_title(text),   # ← عنوان Child بدون شماره
                content=text,              # متن اصلی بدون تغییر
                parent_id=current_parent.id,
            )

        # ---------------- Body ----------------
        else:

            if current_parent is None:
                continue

            if not has_real_child and current_child is None:
                pending_body.append(text)
            else:
                if current_child is not None:
                    current_child.content += "\n" + text

    flush_child()

    if current_parent is not None:

        if not has_real_child:
            flush_pending_as_single_child()

        parents.append(current_parent)

    return parents

In [2]:
from app.utils.Parser import build_chunks
parents = build_chunks(r"/home/son/Desktop/rag/app/data/rag_docs.pdf")

for p in parents:
    print(f"\n■ [parent {p.id}] {p.title[:50]}")
    for c in p.children:
        print(f"  └─ [child {c.id}] {c.title[:50]}")


■ [parent 1] مقدمه و پیشینه
  └─ [child 1.1] مقدمه و پیشینه

■ [parent 2] اجزای اصلی یک سیستم   RAG
  └─ [child 2.1] مقدمه: اجزای اصلی یک سیستم   RAG
  └─ [child 2.2] پردازش و بارگذاری اسناد   (Document Ingestion)
  └─ [child 2.3] تقسیم بندی متن (Chunking)
  └─ [child 2.4] تبدیل به بردار (Embedding)
  └─ [child 2.5] پایگاه داده برداری (Vector Store)
  └─ [child 2.6] بازیابی و رتبه بندی (Retrieval & Reranking)
  └─ [child 2.7] تولید پاسخ (Generation)

■ [parent 3] چالش های RAG برای زبان فارسی
  └─ [child 3.1] مقدمه: چالش های RAG برای زبان فارسی
  └─ [child 3.2] چندریختی متن فارسی
  └─ [child 3.3] اتصال کلمات و توکن بندی
  └─ [child 3.4] کمبود داده های آموزشی
  └─ [child 3.5] راست به چپ بودن متن

■ [parent 4] معیارهای ارزیابی   RAG
  └─ [child 4.1] مقدمه: معیارهای ارزیابی   RAG
  └─ [child 4.2] دقت بازیابی   (Retrieval Accuracy)
  └─ [child 4.3] کیفیت تولید (Generation Quality)
  └─ [child 4.4] صحت استناد   (Faithfulness)

■ [parent 5] معماری های پیشرفته RAG
  └─ [child 5.1] HyDE و روش 

In [8]:
print(parents)

[ParentChunk(id='1', title='۱ .  مقدمه و پیشینه', content='\n\nدر دنیای امروز، مدل های زبانی بزرگ (LLM) به یکی از پایه های اصلی نرم افزارهای هوشمند تبدیل\nشده اند. این مدل ها قادرند متون طبیعی را درک کرده، آن ها را تحلیل کنند و پاسخ هایی دقیق و روان\nتولید نمایند. با این حال، یکی از بزرگ ترین چالش های این مدل ها آن است که دانش آن ها به زمان\nآموزش محدود می شود و دسترسی مستقیمی به داده های جدید، اختصاصی یا سازمانی ندارند.\nبرای رفع این محدودیت، رویکرد «بازیابی-  افزوده تولید» یاRAG (Retrieval-Augmented\nGeneration)   مطرح شد. این رویکرد ترکیبی از دو مرحله اصلی است:\n1.   در مرحله اول، اطالعات مرتبط از یک پایگاه دانش بیرونی بازیابی می شود.\n2.   در مرحله دوم، مدل زبانی از این اطالعات بازیابی شده به عنوان زمینه (Context) استفاده می کند\nتا پاسخ نهایی را تولید نماید.\nمعماری RAG   نخستین بار توسط لوئیس و همکارانش   در سال۲۰۲۰    معرفی شد و از آن زمان تاکنون\nتحوالت گسترده ای را تجربه کرده است. امروزه این معماری در طیف وسیعی از کاربردها، از جمله\nپرسش و پاسخ سازمانی، چت بات های پشتیبانی مشت

In [ ]:
# from app.utils.Retriever import search_children , build_parents_map

# parents_map = build_parents_map(parents)


In [16]:
# parents_map

{'1': ParentChunk(id='1', title='۱ .  مقدمه و پیشینه', content='\n\nدر دنیای امروز، مدل های زبانی بزرگ (LLM) به یکی از پایه های اصلی نرم افزارهای هوشمند تبدیل\nشده اند. این مدل ها قادرند متون طبیعی را درک کرده، آن ها را تحلیل کنند و پاسخ هایی دقیق و روان\nتولید نمایند. با این حال، یکی از بزرگ ترین چالش های این مدل ها آن است که دانش آن ها به زمان\nآموزش محدود می شود و دسترسی مستقیمی به داده های جدید، اختصاصی یا سازمانی ندارند.\nبرای رفع این محدودیت، رویکرد «بازیابی-  افزوده تولید» یاRAG (Retrieval-Augmented\nGeneration)   مطرح شد. این رویکرد ترکیبی از دو مرحله اصلی است:\n1.   در مرحله اول، اطالعات مرتبط از یک پایگاه دانش بیرونی بازیابی می شود.\n2.   در مرحله دوم، مدل زبانی از این اطالعات بازیابی شده به عنوان زمینه (Context) استفاده می کند\nتا پاسخ نهایی را تولید نماید.\nمعماری RAG   نخستین بار توسط لوئیس و همکارانش   در سال۲۰۲۰    معرفی شد و از آن زمان تاکنون\nتحوالت گسترده ای را تجربه کرده است. امروزه این معماری در طیف وسیعی از کاربردها، از جمله\nپرسش و پاسخ سازمانی، چت بات های پشتیبان

In [21]:
from app.utils.Embedder import embed_and_store

count = embed_and_store(parents, collection_name="loader")
print(f"{count} child chunk ذخیره شد")

25 child chunk ذخیره شد


In [8]:
from app.data.Document import RAW_DOCUMENT
from app.models.Chunks import ParentChunk
# from app.utils.Parser import parse_document
from app.utils.Embedder import embed_and_store
from app.utils.Retriever import search_children , build_parents_map
from app.utils.Decider import decide_context , print_decision_result

query = "درباره اجزای rag توضیح بده"
child_results = search_children(query,collection_name="loader" ,top_k=5)
result = decide_context(query, child_results)
print_decision_result(result)

تصمیم      : PARENT
Parent IDs : ['2']
Context نهایی:
------------------------------------------------------------
### [2] اجزای اصلی یک سیستم   RAG

یک سیستم RAG   از چندین مؤلفه کلیدی تشکیل شده است که هر یک نقش مهمی در کیفیت نهایی
پاسخ ها ایفا می کنند.
۲.۱  پردازش و بارگذاری اسناد   (Document Ingestion)
اولین گام در ساخت یک سیستم RAG، آماده سازی و بارگذاری اسناد است. در این مرحله، متن خام از
اسناد استخراج شده، پاک سازی می شود و برای مراحل بعدی آماده می گردد.
اسناد می توانند در قالب های مختلفی مانند PDF   ، Word   ، HTML، متن ساده و حتی پایگاه های داده
ساخت یافته ارائه شوند. پس از استخراج محتوا، داده ها پردازش شده و برای استفاده در مراحل بعدی
سیستم آماده می شوند.
۲.۲  تقسیم بندی متن (Chunking)
پس از بارگذاری اسناد، متن ها به قطعات کوچک تری به نامChunk تقسیم می شوند. این کار به دلیل
محدودیت پنجره زمینه (Context Window) مدل های زبانی ضروری است.
استراتژی های متداول تقسیم بندی عبارت اند از:
الف) تقسیم بندی با اندازه ثابت (Fixed-size Chunking)
در این روش، متن به قطعاتی با تعداد ثابتی از کا

In [25]:
child_results


[ChildSearchResult(score=1.0, child_id='9.1', child_title='آینده    RAG', child_content='آینده RAG به سمت سیستم های پیشرفته تر و هوشمندتر در حال حرکت است، از جمله:\n•   Streaming RAG: تولید پاسخ به صورت تدریجی\n•   Multimodal RAG:   پردازش همزمان متن، تصویر، جدول و نمودار\n•   RAG با حافظه بلندمدت: نگهداری تاریخچه مکالمات و استفاده از آن در پاسخ ها\nبا افزایش ظرفیت مدل های زبانی و گسترش پنجره زمینه (Context Window)  ، برخی کاربردهای\nسنتی RAG ممکن است تغییر کنند، اما در سیستم هایی با داده های بزرگ و نیاز به به ،روزرسانی مداوم\nهمچنان نقش کلیدی خواهد داشت.', parent_id='9', parent_title='آینده    RAG', parent_content='\nآینده RAG به سمت سیستم های پیشرفته تر و هوشمندتر در حال حرکت است، از جمله:\n•   Streaming RAG: تولید پاسخ به صورت تدریجی\n•   Multimodal RAG:   پردازش همزمان متن، تصویر، جدول و نمودار\n•   RAG با حافظه بلندمدت: نگهداری تاریخچه مکالمات و استفاده از آن در پاسخ ها\nبا افزایش ظرفیت مدل های زبانی و گسترش پنجره زمینه (Context Window)  ، برخی کاربردهای\nسنتی RAG ممکن است تغییر ک

In [7]:
result

DecisionResult(decision='PARENT', parent_ids=['5'], child_ids=[], context='### [5] معماری های پیشرفته RAG\n\n۵.۱  HyDE و روش های پیشرفته RAG\nدر رویکردHyDE (Hypothetical Document Embeddings)  ، ابتدا یک سند فرضی بر اساس\nپرسش کاربر تولید می شود و سپس از این سند برای جستجو در پایگاه داده استفاده می گردد. این\nروش باعث بهبود کیفیت بازیابی در مواردی می شود که خود پرسش به تنهایی اطالعات کافی برای\nجستجوی دقیق ندارد.\nعالوه بر HyDE، تکنیک های پیشرفته تری نیز در نسل جدید سیستم های RAG به کار می روند، از\nجمله:\n•   Query Rewriting (بازنویسی پرسش):   بهبود پرسش اولیه برای افزایش دقت بازیابی\n•   Multi-hop Retrieval (بازیابی  چندمرحله ای):   انجام  چند  مرحله  جستجو  برای  پاسخ  به\nپرسش های پیچیده\nFusion of Retrieval Results:   ادغام نتایج چند مدل یا چند منبع بازیابی\n۵.۲  Agentic RAG\nدر این معماری، فرآیند بازیابی به صورت یک چرخه هوشمند و تکرارشونده انجام می شود. یک عامل\n(Agent) تصمیم می گیرد که:\n•    آیا بازیابی الزم است یا خیر\n•    از کدام منابع باید استفاده شود\n•    چند بار فرآیند با

In [1]:
from app.utils.graph import run_query

query = "درباره اجزای rag تقسیم بندی متن رو توضیح بده"

result = run_query(query)



/home/son/Desktop/rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
result["attempts"]

0

In [4]:
result["decision_result"]

DecisionResult(decision='CHILD', parent_ids=[], child_ids=['2.3'], context='### [2.3] تقسیم بندی متن (Chunking)\n۲.۲  تقسیم بندی متن (Chunking)\nپس از بارگذاری اسناد، متن ها به قطعات کوچک تری به نامChunk تقسیم می شوند. این کار به دلیل\nمحدودیت پنجره زمینه (Context Window) مدل های زبانی ضروری است.\nاستراتژی های متداول تقسیم بندی عبارت اند از:\nالف) تقسیم بندی با اندازه ثابت (Fixed-size Chunking)\nدر این روش، متن به قطعاتی با تعداد ثابتی از کاراکترها یا توکن ها تقسیم می شود. این روش ساده و\nسریع است، اما ممکن است جمالت یا پاراگراف ها را از یکدیگر جدا کرده و انسجام معنایی متن را\nکاهش دهد.\nب) تقسیم بندی معنایی (Semantic Chunking)\nدر این رویکرد،  قطعات  بر اساس  مرزهای معنایی مانند  پاراگراف ها،  بخش ها  یا موضوعات  ایجاد\nمی شوند.  این  روش  کیفیت  باالتری  دارد،  زیرا  ارتباط  مفهومی  میان  اجزای  متن  حفظ  می شود،  اما\nپیاده سازی آن پیچیده تر است.\nج) تقسیم بندی با هم پوشانی (Overlapping Chunks)\nبرای حفظ زمینه در مرزهای بین قطعات، از هم پوشانی استفاده می شود. به عنوان مثال، اگر اندا

In [19]:
from app.utils.MultiQueryRetriever import multi_query_retriever
from app.utils.Decider import decide_context, print_decision_result

query = "تولید پاسخ در لحظه چه کمکی میکند؟"

child_results = multi_query_retriever.invoke(
    {
        "question": query
    }
)

result = decide_context(query, child_results, parents_map)

print_decision_result(result)

تصمیم      : CHILD
Child IDs  : ['2.7']
Context نهایی:
------------------------------------------------------------
### [2.7] ۲.۶    تولید پاسخ (Generation)
۲.۶    تولید پاسخ (Generation)
در مرحله نهایی، سؤال اصلی کاربر به همراه قطعات بازیابی شده به عنوانPrompt   به مدل زبانی بزرگ
ارسال می شود.
ساختار Prompt معموال ً  شامل سه بخش است:
1.    دستورالعمل سیستم (System Prompt)
2.   زمینه یا اطالعات بازیابی شده (Retrieved Context)
3.    سؤال کاربر
مدل زبانی با استفاده از این اطالعات، پاسخی جامع، دقیق و مستند تولید می کند.


In [23]:
import fitz
PDF_PATH = "/home/son/Desktop/rag/app/data/rag_docs.pdf"

doc = fitz.open(PDF_PATH)

page = doc[0]

data = page.get_text("dict")

print(data.keys())

dict_keys(['width', 'height', 'blocks'])


In [8]:
for block in data["blocks"]:
    print(block["type"])

0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0


In [11]:
from pprint import pprint
import fitz
PDF_PATH = "/home/son/Desktop/rag/app/data/rag_docs.pdf"

doc = fitz.open(PDF_PATH)

page = doc[0]

data = page.get_text("dict")

pprint(data["blocks"][0])

{'bbox': (442.2699890136719,
          36.922061920166016,
          576.2732543945312,
          65.61934661865234),
 'flags': 0,
 'lines': [{'bbox': (569.6199951171875,
                     36.922061920166016,
                     576.2732543945312,
                     65.61934661865234),
            'dir': (1.0, 0.0),
            'spans': [{'alpha': 255,
                       'ascender': 0.9480000138282776,
                       'bbox': (569.6199951171875,
                                36.922061920166016,
                                576.2732543945312,
                                65.61934661865234),
                       'bidi': 0,
                       'char_flags': 16,
                       'color': 2051705,
                       'descender': -0.48399999737739563,
                       'flags': 0,
                       'font': 'IRKoodak',
                       'origin': (569.6199951171875, 55.91998291015625),
                       'size': 20.040000915527344,
  

In [16]:
import fitz

doc = fitz.open(PDF_PATH)

page = doc[0]

data = page.get_text("dict")

for block in data["blocks"]:
    for line in block["lines"]:
        for span in line["spans"]:
            print(
                f"{span['size']:5.1f}",
                span["font"],
                repr(span["text"])
            )

 20.0 IRKoodak '۱'
 20.0 IRKoodak ' .'
 20.0 IRKoodak ' مقدمه و پیشینه '
 16.0 IRKoodak 'د'
 16.0 IRKoodak 'ر دنیای امروز، مدل های زبانی بزرگ (LLM) به یکی از پایه'
 16.0 IRKoodak 'های اصلی نرم افزارهای هوشمند تبدیل'
 16.0 IRKoodak 'شده'
 16.0 IRKoodak 'اند. این مدل'
 16.0 IRKoodak 'ها قادرند متون طبیعی را درک کرده، آن'
 16.0 IRKoodak 'ها را تحلیل کنند و پاسخ هایی دقیق و روان'
 16.0 IRKoodak ' تولید نمایند. با این حال، یکی از بزرگ'
 16.0 IRKoodak 'ترین چالش'
 16.0 IRKoodak 'های این مدل'
 16.0 IRKoodak 'ها آن است که دانش آن ها به زمان'
 16.0 IRKoodak 'آموزش محدود می'
 16.0 IRKoodak 'شود و دسترسی مستقیمی به داده'
 16.0 IRKoodak 'های جدید، اختصاصی یا سازمانی ندارند.'
 16.0 IRKoodak ' '
 16.0 IRKoodak ' '
 16.0 IRKoodak 'برای رفع این محدودیت، رویکرد «بازیابی-'
 16.0 IRKoodak ' افزوده تولید» یاRAG (Retrieval-Augmented '
 16.0 IRKoodak 'Generation) '
 16.0 IRKoodak ' مطرح شد. این رویکرد ترکیبی از دو مرحله اصلی است:'
 16.0 IRKoodak ' '
 16.0 IRKoodak '1.'
 16.0 ArialMT ' '
 16.0 IRKoodak 'در م

In [12]:
block = data["blocks"][0]

print(block.keys())

dict_keys(['type', 'number', 'flags', 'bbox', 'lines'])


In [13]:
print(len(block["lines"]))

3


In [14]:
line = block["lines"][0]

print(line.keys())

dict_keys(['spans', 'wmode', 'dir', 'bbox'])


In [15]:
span = line["spans"][0]

print(span)

{'size': 20.040000915527344, 'flags': 0, 'bidi': 0, 'char_flags': 16, 'font': 'IRKoodak', 'color': 2051705, 'alpha': 255, 'ascender': 0.9480000138282776, 'descender': -0.48399999737739563, 'text': '۱', 'origin': (569.6199951171875, 55.91998291015625), 'bbox': (569.6199951171875, 36.922061920166016, 576.2732543945312, 65.61934661865234)}
